In [2]:
import os
import time
import re
import numpy as np
from scipy import stats as st
from functools import reduce
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from Bio.Align import PairwiseAligner as pairwise2
from Bio.Seq import Seq

In [ ]:
# ccs --num-threads 50 --min-passes 3 --min-rq 0.99 subreads.bam subreads.ccs.bam
# lima -j 40 --same --ccs --min-score 80 --min-end-score 50 --min-ref-span 0.75 --split-named subreads.ccs.bam barcode.fasta subreads.ccs.lima.bam

# function

In [393]:
# 获取序列的CDS
def get_ref_seq_CDS(ref_seq_raw):
    read_frame = {0:'', 1:'',2:''}
    stop_codon = ['TAA', 'TAG', 'TGA']
    for frame_start_site in [0,1,2]:
        in_frame = 0
        start_site = frame_start_site
        while start_site+3 < len(ref_seq_raw):
            triplet = ref_seq_raw[start_site:start_site+3]
            if triplet == 'ATG':
                in_frame = 1
            if in_frame:
                read_frame[frame_start_site] = read_frame[frame_start_site] + triplet
                if triplet in stop_codon:
                    break
            start_site +=3
    ref_seq = ''
    for i in read_frame:
        seq = read_frame[i]
        if len(seq) > len(ref_seq):
            ref_seq = seq
    return ref_seq.upper()
# 获得密码子df
def get_df_codon(codon_path):
    lines = []
    with open(codon_path, 'r') as f:
        lines = f.readlines()
    new_line = ''
    for line in lines[3:-3]:
        if line == '\n':
            continue
        new_line += line[:-1].replace('(', '').replace(')', '').replace('   ', ' ').replace('  ', ' ') + ' '
    fields = ['triplet', 'amino_acid', 'fraction', 'frequency', 'number']
    df = pd.DataFrame(columns=fields)
    items = new_line[:-1].split(' ')
    for i in range(64):
        item = items[i*5:(i+1)*5]
        df = pd.concat([df, pd.DataFrame({'triplet':[item[0]], 'amino_acid':[item[1]], 'fraction':[item[2]], 'frequency':[item[3]], 'number':[item[4]]})], ignore_index=True)
    return df

def get_codon_dict(df_codon):
    codon_dict = {}  # {triplet: amino_acid}
    for i in df_codon.index:
        triplet = df_codon.loc[i, 'triplet']
        amino_acid = df_codon.loc[i, 'amino_acid']
        codon_dict[triplet] = amino_acid
    return codon_dict

def mut2triplet(mut, ref_seq):
    
    base_index = int(mut[1:-1]) -1
    triplet_index = int(base_index/3)
    raw_triplet = ref_seq[triplet_index*3:triplet_index*3+3]
    
    triplet_base_list = list(raw_triplet)
    triplet_base_list[int(mut[1:-1])%3 -1] = mut[-1]
    new_triplet = ''.join(triplet_base_list)
    return raw_triplet, new_triplet

def is_nonsense_mutation(new_triplet, raw_triplet, choose_codon_dict=0):
    if choose_codon_dict == 0:
        codon_dict = {'UUU': 'F', 'UCU': 'S', 'UAU': 'Y', 'UGU': 'C', 'UUC': 'F', 'UCC': 'S', 'UAC': 'Y', 'UGC': 'C', 'UUA': 'L', 'UCA': 'S', 'UAA': '*', 'UGA': '*', 'UUG': 'L', 'UCG': 'S', 'UAG': '*', 'UGG': 'W', 'CUU': 'L', 'CCU': 'P', 'CAU': 'H', 'CGU': 'R', 'CUC': 'L', 'CCC': 'P', 'CAC': 'H', 'CGC': 'R', 'CUA': 'L', 'CCA': 'P', 'CAA': 'Q', 'CGA': 'R', 'CUG': 'L', 'CCG': 'P', 'CAG': 'Q', 'CGG': 'R', 'AUU': 'I', 'ACU': 'T', 'AAU': 'N', 'AGU': 'S', 'AUC': 'I', 'ACC': 'T', 'AAC': 'N', 'AGC': 'S', 'AUA': 'I', 'ACA': 'T', 'AAA': 'K', 'AGA': 'R', 'AUG': 'M', 'ACG': 'T', 'AAG': 'K', 'AGG': 'R', 'GUU': 'V', 'GCU': 'A', 'GAU': 'D', 'GGU': 'G', 'GUC': 'V', 'GCC': 'A', 'GAC': 'D', 'GGC': 'G', 'GUA': 'V', 'GCA': 'A', 'GAA': 'E', 'GGA': 'G', 'GUG': 'V', 'GCG': 'A', 'GAG': 'E', 'GGG': 'G'} 
    else:
        codon_dict = choose_codon_dict
    new_triplet_rna = new_triplet.replace('T', 'U')
    raw_triplet_rna = raw_triplet.replace('T', 'U')
    if codon_dict[new_triplet_rna] == codon_dict[raw_triplet_rna]:
        return 1
    else:
        return 0

def base_correct_rate(x, phred=33):
    return 1-np.power(10, -((ord(x) - phred)/10))

def muts2aamuts_str(muts, ref_seq):
    if muts == '':
        return ''
    if muts == 'unmatch':
        return 'unmatch'
    aamuts = ''
    aamuts_num = 0
    for mut in muts.split(':'):
        aa_site = int((int(mut[1:-1])-1)/3) + 1
        try:
            raw_triplet, new_triplet = mut2triplet(mut, ref_seq)
        except:
            input(f'{mut}')
        raw_aa = codon_dict[raw_triplet.replace('T', 'U')]
        new_aa = codon_dict[new_triplet.replace('T', 'U')]
        if new_aa != raw_aa:
            aamuts += f'{raw_aa}{aa_site}{new_aa}:'
            aamuts_num +=1
        if new_aa == '*':
            break
    if len(aamuts) >0:
        if aamuts[-1] == ':':
            aamuts = aamuts[:-1]
    return aamuts

def fastq2df(fastq_path):
    lines = []
    with open(fastq_path, 'r') as f:
        _lines = f.readlines()
    for line in _lines:
        lines.append(line.replace('\n', ''))
    
    seq_lens = []
    for seq in lines[1::4]:
        seq_lens.append(len(seq))
    
    df = pd.DataFrame({'read_info':lines[::4], 
                       'seq_len':seq_lens,
                       'seq':lines[1::4], 
                       'fastq_quality':lines[3::4], })
    return df

def load_alignresult(df, ref_seq, alignresult_path, base_quality_threshold=0):

    result_dict = {}

    result_df = pd.read_json(alignresult_path)
    for i in result_df.index:

        read_info, results, results_2 = result_df.loc[i, 'read_info'], result_df.loc[i, 'muts_0'], result_df.loc[i, 'muts_1']   
        read_quality = df.loc[df['read_info']==read_info, 'fastq_quality'].item()
        
        '''result_dict[read_info] = {
            'mut':[], 'ns_mut':[], 'ms_mut':[], 'aa_mut':[], 'insert':[], 'delete':[],
            'mut_num':0, 'ns_mut_num':0, 'ms_mut_num':0, 'aa_mut_num':0, 'insert_num':0, 'delete_num':0,}'''

        result_dict[read_info] = {'ns_mut':[], 'ns_mut_num':0, 
                                  'ms_mut':[], 'ms_mut_num':0, 
                                  'aa_mut':[], 'aa_mut_num':0,
                                  'insert_raw':[], 'insert_raw_num':0,
                                  'delete_raw':[], 'delete_raw_num':0,}

        if results == ['unmatch']:
            for item in result_dict[read_info]:
                result_dict[read_info][item] == 'unmatch'
        else:
            mut_in_result2_index = 0
            for mut in results:
                if mut == '':
                    continue
                # 缺失
                elif mut[-1] == '-':
                    result_dict[read_info]['delete_num'] +=1
                    result_dict[read_info]['delete'].append([mut])
                    mut_in_result2_index -=1
                else:
                    mut_B = results_2[mut_in_result2_index]
                    site_B = int(mut_B[:-1])
                    base_quality = base_correct_rate(read_quality[site_B-1])

                    if base_quality < base_quality_threshold:
                        mut_in_result2_index +=1
                        continue
                    # 插入
                    if mut[0] == '-':
                        result_dict[read_info]['insert_num'] +=1
                        result_dict[read_info]['insert'].append([mut])
                    # 替换
                    else:
                        raw_triplet, new_triplet = mut2triplet(mut, ref_seq)
                        if len(raw_triplet) <3:
                            continue
                        #result_dict[read_info]['mut_num'] +=1
                        #result_dict[read_info]['mut'].append([mut])
                        if is_nonsense_mutation(new_triplet, raw_triplet):
                            result_dict[read_info]['ns_mut_num'] +=1
                            result_dict[read_info]['ns_mut'].append([mut])
                        else:
                            result_dict[read_info]['ms_mut_num'] +=1
                            result_dict[read_info]['ms_mut'].append([mut])

                            aa_mut = muts2aamuts_str(mut, ref_seq)
                            result_dict[read_info]['aa_mut_num'] +=1
                            result_dict[read_info]['aa_mut'].append([aa_mut, mut])
                mut_in_result2_index +=1
    
    for i in df.index:
        read_info = df.loc[i, 'read_info']
        for kw in result_dict[read_info]:
            df.loc[i, kw] = ''
            df.at[i, kw] = result_dict[read_info][kw]

    return df
# seq len quality control
def df_quality_control(df):
    df_new = {}
    for sample in df:
        df_new[sample] = {}
        
        _df = df[sample]['info']
        
        #mean_len = _df['seq_len'].mean()
        #a = _df['seq_len'].std()
        #min_len = mean_len - a*1.96  # 95%
        #max_len = mean_len + a*1.96
        min_len, max_len = st.norm.interval(confidence=0.90, loc=_df['seq_len'].mean(), scale=_df['seq_len'].std())
        
        _df = _df.loc[_df['seq_len']>=min_len].loc[_df['seq_len']<=max_len]
        _df = _df.loc[_df['ms_mut']!='unmatch']
        
        df_new[sample]['info'] = _df
    return df_new

def cut_seq_by_special_len(ref_seq, special_len=3):
    triple_list = []
    for i in range(len(ref_seq[0::special_len])):
        triple_list.append(ref_seq[i*special_len:(i+1)*special_len])
    return triple_list

def num2percentage_for_dict(in_dict, max_num=0): 
    total = 0
    if max_num==0:
        for i in in_dict:
            total += in_dict[i]
    else:
        total = max_num
    new_dict = {}
    for i in in_dict:
        new_dict[i] = in_dict[i]/total
    return new_dict

def dict_count(in_dict, k, num=1):
    try:
        in_dict[k] = in_dict[k] + num
    except:
        in_dict[k] = num
    return in_dict

def dot_split(number):
    number = str(number)
    if '.' in number:
        number_split = number.split('.')
        if len(number_split[1]) == 1:
            number = number + '0'
        elif len(number_split[1]) > 2:
            number = str(round(float(number), 2))
    else:
        number = number + '.00'
    return float(number)

def get_number_converting_dict(toHAtype=3):
    convert_dict = {'toHAtype':toHAtype, }
    df_numbering = pd.read_excel('assets/toH5H3numbering.xlsx')
    for i in df_numbering.index:
        query_num = df_numbering.loc[i, 'Query_num']
        ha_num = df_numbering.loc[i, f'H{toHAtype}_num']
        if query_num != '-':
            convert_dict[query_num] = ha_num
    return convert_dict

def main(path, ref_raw_dict, df=0, special_sample=[], load_json=1, base_quality_threshold=0):
    if df == 0:
        df = {}
    
    for fastq in os.listdir(path):

        # .fastq
        if not fastq.endswith('.fastq'):
            continue
        fastq_path = f'{path}/{fastq}'
        
        # .alignresult
        alignresult_path = f'{fastq_path}.alignresult.json'
        if not os.path.exists(alignresult_path):
            continue
        
        sample = fastq.split('.fastq')[0]
        if sample in df:
            continue
        
        if (special_sample != []) and (sample not in special_sample):
            continue
        
        df[sample] = {}
        time0 = time.time()
        print(f'fastq2df: {sample}')
    
        # load .json and continue if exists
        if load_json and (os.path.exists(f'{path}/{sample}.info.json')):
            df[sample]['info'] = pd.read_json(f'{path}/{sample}.info.json')
            print(f'Load df[{sample}][info] from {sample}.info.json')
        else:
            df[sample]['info'] = fastq2df(fastq_path)
            print(f'load_alignresult: {sample}')
                
            if sample[:2] == 'C-':
                protein_name = sample.split('-')[1]
                ref_seq = get_ref_seq_CDS(ref_raw_dict[protein_name])
                print(protein_name)
            else:
                ref_seq = get_ref_seq_CDS(ref_raw_dict['HA'])
                print('HA')
                
            ref_seq = ref_seq.upper()
            
            df[sample]['info'] = load_alignresult(df[sample]['info'], ref_seq, alignresult_path, base_quality_threshold)  
            df[sample]['info'].to_json(f'{path}/{sample}.info.json')
            print(f'{sample} ok.  Time: {time.time()-time0}\n')

    # quality control
    df = df_quality_control(df)
        
    return df

# main

In [ ]:
# 2 execute main functions

df_human_codon = get_df_codon('assets/codon_human.txt')
codon_dict = get_codon_dict(df_human_codon)

ref_raw_dict = {
    'HA':'AGCAAAAGCAGGGGAATTTCACAACCACTCAAGATGGAGACAGTATTACTAATAACTATACTAGTAGTAGCAACAGTGAGCAATGCAGATAAGATCTGCATCGGCTATCAATCAACGAACTCCACGGAAACTGTGGACACACTAACAGAAAACAATGTCCCTGTGACACATGCCAAAGAACTGCTCCACACAGAGCATAATGGGATGCTGTGTGCAACAAGCTTGGGACACCCTCTTATTCTAGACACCTGCACCATTGAAGGGCTAATTTATGGCAATCCTTCTTGTGATCCATTGCTGGGAGGAAGAGAATGGTCCTATATCGTCGAGAGGCCATCAGCTGTTAACGGATTATGTTATCCCGGGAATGTAGAAAATCTAGAAGAGCTAAGATCACTTTTTAGTTCTTCTAGGTCTTATCAAAGGATCCAGATCTTCCCCGACACAATCTGGAATGTGTCTTACAGTGGGACAAGCAAAGCATGCTCGGATTCATTCTACAGAAGCATGAGATGGTTGACTCAAAAGAACAACGCTTACCCTACCCAAGATGCTCAATACACAAATAATCAAGGAAAGAACATTCTTTTCATGTGGGGTATAAATCATCCACCCACCGATACTGTGCAGACAGATCTGTACACCAGAACCGACACAACAACGAGTGTGGCAACAGAGGAAATGAATAGGATCTTTAAACCATTGATAGGACCAAGGCCCCTTGTCAACGGTTTGATgGGAAGAATTAATTaTTATTGGTCAGTaTTGAAACCGGGCCAAACACTGCGGATAAAATCTGATGGGAATCTAATAGCTCCATGGTATGGACACATTCTTTCAGGAGAGAGCCACGGAAGAATCCTAAAGACTGATTTAAAAAGGGGTAGCTGCACAGTGCAGTGTCAAACAGAGAAAGGTGGCTTAAACACAACATTGCCCTTCCAAAaTGTAAGTAAGtATGCATTTGGAAAcTGCTCAAAGtACATTGGTGTAAAGAGTCTCAAACTTGCAGTTGGTCTGAGGAATGTGCCTTCTAGATCTAGTAGAGGACTATTCGGGGCCATAGCAGGATTTATAGAGGGAGGTTGGTCAGGACTAGTTGCTGGTTGGTATGGGTTCCAGCATTCAAATGACCAAGGGGTTGGTATGGCAGCAGATAGAGATTCAACCCAAAAGGCAATTGATAAAATAACATCCAAAGTGAATAATATAGTCGACAAAATGAACAAGCAGTATGAAATTATTGATCATGAATTCAGTGAGGTAGAAACTAGGCTTAACATGATCAATAATAAGGTTGATGATCAAATCCAAGATATATGGGCATATAATGCAGAATTGCTAGTTCTGCTTGAAAACCAGAAAACACTCGATGAACATGACGCGAATGTAAACAATCTATATAATAAAGTGAAGAGGGCGTTGGGTTCCAATGCGGTAGAAGATGGGAAAGGATGTTTCGAGCTATACCATAAATGTGATGACCATTGCATGGAGACAATTCGGAATGGGACCTACAACAGGAGGAAGTATCAAGAGGAATCAAAATTAGAAAGACAGAAAATAGAGGGGGTAAAGCTGGAATCGGAAGAAACTTACAAAATCCTCACCATTTATTCGACTGTCGCCTCATCTCTTGTGATTGCAATGGGGTTTGCTGCCTTTTTGTTCTGGGCCATGTCCAATGGGTCTTGCAGATGCAACATTTGTATATAATTGGCAAAAACACCCTTGTTTCTACT',
}

path = 'data/fastq'

print('start\n')

#1 get df

dfs = main(path, ref_raw_dict, df=0, special_sample=[], load_json=1, base_quality_threshold=0.9)

# Time: <8min

In [559]:
def delete_seq_process(df_info, ref_seq):

    # delete_seq
    for i in df_info.index:

        # 1.delete_seq_raw
        delete_raw = []
        deletes = df_info.loc[i, 'delete_raw']
        pre_triple = []
        for delete in deletes:
            site = int(delete[1:-1])
            if len(pre_triple) == 0:
                pre_triple.append(delete)
                pre_delete_site = site
                continue
            if site == (int(pre_triple[-1][1:-1]) + 1):
                pre_triple.append(delete)
                pre_delete_site = site
            else:
                if len(pre_triple) >= 3:
                    delete_raw.append(pre_triple) # 保存
                # 重置
                pre_triple = [delete]
                pre_delete_site = site
        # 最后一个
        if len(pre_triple) >= 3:
            delete_raw.append(pre_triple)

        # 2.delete_seq
        delete_seqs = []
        delete_num = 0
        for delete_seq in delete_raw:
            if 'A1-' in delete_seq: # 第一个
                continue
            if f'{ref_seq[-1]}{len(ref_seq)}-' in delete_seq: # 最后一个
                continue
            delete_seqs.append(delete_seq)
            delete_num += len(delete_seq)

            df_info.loc[i, 'delete'] = ''
            df_info.at[i, 'delete'] = delete_seqs
            df_info.loc[i, 'delete_num'] = delete_num

def insert_seq_process(df_info):
    # insert_seq
    for i in df_info.index:

        # 1.insert_seq_raw
        insert_seq_raw = []
        inserts = df_info.loc[i, 'insert_raw']
        insert_seq = ''
        for insert in inserts:
            site = int(insert[1:-1])
            base = insert[-1]
            if len(insert_seq) == 0:
                insert_seq = base
                pre_site = site
                continue
            if site == pre_site:
                insert_seq += base
            else:
                if len(insert_seq) >=3:
                    insert_seq_raw.append(f'{pre_site}-{insert_seq}') # example: 111-AATTCCGG
                insert_seq = base
                pre_site = site
        # 最后一个
        if len(insert_seq) >=3:
            insert_seq_raw.append(f'{pre_site}-{insert_seq}')

        # 2.insert_seq
        insert_seqs = []
        insert_num = 0
        for insert_seq in insert_seq_raw:
            insert_num += len(insert_seq.split('-')[1])
        
        df_info.loc[i, 'insert'] = ''
        df_info.at[i, 'insert'] = insert_seq_raw
        df_info.loc[i, 'insert_num'] = insert_num

def func(df_info):
    for i in df_info.index:

        try:
            if ['T700-', 'T701-', 'G702-'] in df_info.loc[i, 'delete']:
                df_info.loc[i, 'aa_mut'].append('L234-')
                df_info.loc[i, 'aa_mut_num'] += 1
        except:
            pass

        try:
            if ['T701-', 'T702-', 'G703-'] in df_info.loc[i, 'delete']:
                df_info.loc[i, 'aa_mut'].append('M235-')
                df_info.loc[i, 'aa_mut_num'] += 1
        except:
            pass
        
        try:
            if '1012-AGGAGAAGAAAG' in df_info.loc[i, 'insert']:
                df_info.loc[i, 'aa_mut'].append('-337RRRK')
                df_info.loc[i, 'aa_mut_num'] += 4
        except:
            pass

for sample in dfs:
    #insert_seq_process(dfs[sample]['info'])
    #delete_seq_process(dfs[sample]['info'], ref_seq=get_ref_seq_CDS(ref_raw_dict['HA']))
    func(dfs[sample]['info'])

# Time: <30min

In [594]:
# df[sample][ms_mut, ns_mut, aa_mut]
def get_value_counts(df, column_list):
    for sample in df:
        for column in column_list:
            value_counts = df[sample]['info'][column].value_counts()
            value_counts = pd.DataFrame({'mut':value_counts.index, 'mut_num':[len(i) for i in value_counts.index], 'read_num':value_counts.values})
            df[sample][column] = value_counts

get_value_counts(dfs, ['ms_mut', 'ns_mut', 'aa_mut', 'delete', 'insert'])

In [ ]:
# 将df_info保存成json文件
def save_df_info(df):
    for sample in df:
        df[sample]['info'].to_json(f'{path}/{sample}.info.json')
        print(f'{sample} ok.')

save_df_info(dfs)